<a href="https://colab.research.google.com/github/zabdielVR/IA_with_collab/blob/main/Copy_of_Hands_On_Prompt_Engineering_y_Sistemas_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **HANDS - ON: PROMPT ENGINEERING Y SISTEMAS RAG**

Una vez vista la masterclass ***Prompt Engineering y Sistemas RAG***, se proporciona el siguiente ***Colab*** para construir, en vivo, distintas estrategias de prompting y un mini sistema de RAG.

Usamos **Groq** para tener acceso a inferencia con GPU gratuita, sin necesidad de infraestructura propia.

## **CONFIGURACIÓN DEL ENTORNO**

Para no exponer tu ***API key*** directamente en el código, Colab ofrece un panel de ***Secrets*** (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la ***API key*** para guardarla dentro de una variable y usarla dentro del notebook.

In [2]:
# Instalar cliente de Groq y leer API key desde Colab Secrets

!pip install groq --quiet

from groq import Groq
from google.colab import userdata

client = Groq(api_key=userdata.get('GROQ_API_KEY'))
print("Cliente de Groq inicializado correctamente.")

Cliente de Groq inicializado correctamente.


## **ESTRATEGIAS DE PROMPTS**

### **ZERO-SHOT VS. FEW-SHOT**

Zero-shot es pedirle al modelo que haga algo sin darle ejemplos. Few-shot le muestra dos o tres ejemplos de entrada y salida antes de pedirle la tarea real. Vamos a comparar ambas estrategias con la misma tarea de clasificación.

In [39]:
# Prompt de clasificación en modo zero-shot

prompt_zero_shot = "Clasifica el sentimiento de esta reseña en Positivo, Negativo o Mixto: 'El envío llegó tarde pero el producto es excelente.' Respuesta muy breve y corta."

response_zero = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_zero_shot}]
)

print("Zero-shot:", response_zero.choices[0].message.content)

Zero-shot: Mixto.


In [40]:
# Prompt de clasificación en modo few-shot

prompt_few_shot = """Clasifica el sentimiento de cada reseña como Positivo, Negativo o Mixto.

Reseña: "Me encantó, llegó rápido y en perfecto estado."
Sentimiento: Positivo

Reseña: "Nunca llegó mi pedido, pésimo servicio."
Sentimiento: Negativo

Reseña: "El envío llegó tarde pero el producto es excelente."
Sentimiento:"""

response_few = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_few_shot}]
)

print("Few-shot:", response_few.choices[0].message.content)
# El formato few-shot suele acotar mejor la salida a una sola palabra de la categoría esperada

Few-shot: Sentimiento: Mixto


### **CHAIN-OF-THOUGHT**

Chain-of-thought le pide al modelo mostrar su razonamiento paso a paso antes de la respuesta final, algo que mejora notablemente el desempeño en problemas de lógica.

In [41]:
# Razonamiento paso a paso (chain-of-thought)

problema = (
    "Un tren sale de la ciudad A a 80 km/h. Dos horas después, otro tren sale de la misma "
    "ciudad hacia el mismo destino a 120 km/h. ¿Cuánto tiempo tarda el segundo tren en "
    "alcanzar al primero? Muestra tu razonamiento paso a paso antes de dar la respuesta final."
)

response_cot = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": problema}]
)

print(response_cot.choices[0].message.content)
# problema_directo = problema.split(".")[-2]  # variante: pedir solo el número, sin razonamiento, y comparar qué tan seguido se equivoca

**Paso 1: Calcular la distancia que el primer tren ya tiene cuando sale el segundo**

- Velocidad del primer tren: \(v_1 = 80\ \text{km/h}\)
- Tiempo que viaja antes de que salga el segundo tren: \(t_{\text{h}1} = 2\ \text{h}\)

\[
\text{Distancia adelantada} = v_1 \times t_{\text{h}1}
= 80\ \text{km/h} \times 2\ \text{h}
= 160\ \text{km}
\]

El primer tren ya está 160 km adelante cuando el segundo comienza su recorrido.

---

**Paso 2: Determinar la velocidad relativa**

- Velocidad del segundo tren: \(v_2 = 120\ \text{km/h}\)
- Velocidad del primero: \(v_1 = 80\ \text{km/h}\)

\[
\text{Velocidad relativa} = v_2 - v_1 = 120\ \text{km/h} - 80\ \text{km/h} = 40\ \text{km/h}
\]

El segundo tren gana terreno a razón de 40 km por hora.

---

**Paso 3: Calcular el tiempo que le toma alcanzar al primero**

\[
t_{\text{atrapar}} = \frac{\text{Distancia adelantada}}{\text{Velocidad relativa}}
= \frac{160\ \text{km}}{40\ \text{km/h}}
= 4\ \text{h}
\]

---

### Respuesta

El segundo tren alcanza

### **EL LÍMITE DEL PROMPT: LO QUE EL MODELO NUNCA VIO**

Ninguna técnica de prompting le da información nueva al modelo. Si le preguntamos algo que no pudo haber visto en su entrenamiento, puede alucinar una respuesta que suene convincente pero no sea real.

In [42]:
# Preguntar algo que el modelo no pudo haber visto en su entrenamiento y observar si alucina

prompt_desconocido = (
    "¿Cuál fue el resultado del ganador del mundial de 2026."
)

response_alucinacion = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_desconocido}]
)

print(response_alucinacion.choices[0].message.content)
# Por más segura que suene la respuesta, el modelo no tiene forma de saber esto: es una alucinación

No hay resultados disponibles para el ganador del Mundial 2026, porque el torneo todavía no se ha celebrado. La Copa Mundial de la FIFA 2026 está programada para disputarse en Estados Unidos, Canadá y México, y su final está prevista para finales de 2026. Por lo tanto, la información sobre el resultado y el campeón aún no está disponible.


## **RAG: BUSCAR ANTES DE RESPONDER**

RAG separa el proceso en dos pasos: primero un sistema de búsqueda encuentra los fragmentos más relevantes de una base de conocimiento propia (usando *embeddings* y similitud semántica); después, esos fragmentos se le entregan al modelo junto con la pregunta para generar la respuesta.

In [9]:
# Instalar sentence-transformers

!pip install sentence-transformers --quiet

from sentence_transformers import SentenceTransformer
import numpy as np

In [43]:
# Definir la base de conocimiento (política de devoluciones) y generar sus embeddings
from sentence_transformers.util import cos_sim

modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

documentos = [
"AWS Lambda permite ejecutar código sin administrar servidores.",
    "Amazon S3 es un servicio de almacenamiento de objetos.",
    "RAG combina recuperación de información con modelos generativos.",
    "Amazon RDS permite utilizar bases de datos relacionales administradas."
]

# Pregunta
pregunta = "¿Qué servicio permite ejecutar código sin servidores?"

# Vectorizar fragmentos y pregunta
embeddings_fragmentos = modelo_embeddings.encode_document(documentos)
embedding_pregunta = modelo_embeddings.encode_query(pregunta)

# Calcular similitud coseno
similitudes = cos_sim(
    embedding_pregunta,
    embeddings_fragmentos
)

# Mostrar similitud de cada fragmento
for i, score in enumerate(similitudes[0]):
    print(f"Fragmento {i + 1}: {fragmentos[i]}")
    print(f"Similitud: {score.item():.4f}\n")

# Obtener el fragmento más relevante
indice_mayor = similitudes.argmax().item()

fragmento_relevante = documentos[indice_mayor]
score_mayor = similitudes[0][indice_mayor].item()

print("=== FRAGMENTO MÁS RELEVANTE ===")
print(fragmento_relevante)
print(f"Similitud: {score_mayor:.4f}")

embeddings_documentos = modelo_embeddings.encode(documentos)
print("Embeddings generados:", embeddings_documentos.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Fragmento 1: AWS Lambda permite ejecutar código sin administrar servidores.
Similitud: 0.7159

Fragmento 2: Amazon S3 es un servicio de almacenamiento de objetos.
Similitud: 0.1474

Fragmento 3: RAG combina recuperación de información con modelos generativos.
Similitud: 0.1219

Fragmento 4: Amazon RDS permite utilizar bases de datos relacionales administradas.
Similitud: 0.2914

=== FRAGMENTO MÁS RELEVANTE ===
AWS Lambda permite ejecutar código sin administrar servidores.
Similitud: 0.7159
Embeddings generados: (4, 384)


In [44]:
# Definir una función que calcule la similitud entre la pregunta y cada fragmento, y regrese el más relevante

def buscar_fragmento(pregunta):
    embedding_pregunta = modelo_embeddings.encode([pregunta])
    similitudes = np.dot(embeddings_documentos, embedding_pregunta.T).flatten()
    indice_mas_similar = np.argmax(similitudes)
    return documentos[indice_mas_similar]

pregunta = "¿Qué es AWS?"
fragmento = buscar_fragmento(pregunta)
print("Fragmento recuperado:", fragmento)

Fragmento recuperado: Amazon S3 es un servicio de almacenamiento de objetos.


In [45]:
# Enviar la pregunta junto con el fragmento recuperado al modelo y mostrar la respuesta con RAG

prompt_rag = f"""


Responde únicamente utilizando la información contenida en el contexto proporcionado.

Reglas:
1. No utilices conocimiento externo.
2. No inventes información.
3. Si la respuesta no se encuentra en el contexto, responde:
  'No encontré esa información en la documentación proporcionada.'
4. No hagas suposiciones.
5. Basa cada respuesta exclusivamente en los fragmentos recuperados del documento.

Contexto: {fragmento}

Pregunta: {pregunta}

Respuesta muy breve y corta."""

response_rag = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_rag}]
)
print("=== RESPUESTA CON  RAG ===")
print(response_rag.choices[0].message.content)



# -------------------------
# RESPUESTA SIN RAG
# -------------------------

response_sin_rag = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": pregunta
        }
    ]
)

print("=========================================================")
print("=== RESPUESTA SIN RAG ===")
print(response_sin_rag.choices[0].message.content)
# Compara esta respuesta contra lo que el modelo diría sin el fragmento: sin RAG probablemente improvisaría una política genérica

=== RESPUESTA CON  RAG ===
No encontré esa información en la documentación proporcionada.
=== RESPUESTA SIN RAG ===
**AWS – Amazon Web Services**  

AWS es la plataforma de computación en la nube de Amazon, una división de Amazon.com que proporciona una amplia gama de servicios de infraestructura, plataforma y software bajo demanda. Fue lanzada en 2006 y, desde entonces, se ha convertido en el proveedor de nube pública más grande del mundo.

### ¿Qué ofrece AWS?

| Categoría | Ejemplos de servicios | Qué hace |
|-----------|----------------------|----------|
| **IaaS (Infraestructura como Servicio)** | EC2, Lightsail, EBS, VPC | Máquinas virtuales, almacenamiento en bloque, redes virtuales. |
| **PaaS (Plataforma como Servicio)** | RDS, Aurora, DynamoDB, Elastic Beanstalk, ECS, EKS | Bases de datos gestionadas, contenedores, aplicaciones sin preocuparse de la capa subyacente. |
| **SaaS (Software como Servicio)** | Amazon Connect, QuickSight, WorkSpaces | Herramientas listas para usar 